# Simulated Confocal Microscopy Volumes and Particle Detection Metrics for Dense Colloidal Suspensions Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In this notebook, all dataset entities are referenced by their Croissant `@id` values whenever available. Because Croissant datasets can vary in structure, the notebook also includes defensive inspection code to discover record sets and fields dynamically before loading them into pandas DataFrames.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We start by creating a dataset object from the Croissant JSON-LD URL and printing a compact metadata summary.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.ywha-byn8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

In [ ]:
# Optional: inspect the metadata as JSON for schema-level exploration.
# This converts the metadata object to a JSON-like structure without subscripting dataset.metadata directly.
metadata_json = metadata.to_json()
print(json.dumps({
    "@id": metadata_json.get("@id"),
    "name": metadata_json.get("name"),
    "description": metadata_json.get("description"),
    "datePublished": metadata_json.get("datePublished"),
    "keywords": metadata_json.get("keywords", [])[:5]
}, indent=2))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR^2 package excerpt included with this prompt shows an empty `recordSet` list at the top level. In practice, some Croissant datasets expose record sets directly, while others emphasize file distributions and schema metadata.

The next cells inspect the loaded dataset object defensively to discover:
- record set `@id` values,
- field `@id` values,
- and a small sample of records if available.

This keeps the notebook usable even when the exact internal structure is not known in advance.

In [ ]:
# Inspect metadata JSON keys to understand the Croissant package structure.
print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

In [ ]:
# Try to discover record sets from the metadata JSON.
record_sets_json = metadata_json.get("recordSet", []) or metadata_json.get("recordSets", []) or []

record_set_ids = []
record_set_field_map = {}

for rs in record_sets_json:
    rs_id = rs.get("@id", "<missing_record_set_id>")
    record_set_ids.append(rs_id)
    fields = rs.get("field", []) or rs.get("fields", []) or []
    field_ids = [field.get("@id", "<missing_field_id>") for field in fields]
    record_set_field_map[rs_id] = field_ids

print("Discovered record set @id values:")
print(record_set_ids if record_set_ids else "No record sets were declared in top-level metadata.")

if record_set_field_map:
    print("\nField @id values by record set:")
    pprint(record_set_field_map)

If record sets are declared, the following cell previews a few records from each record set using its `@id`.

In [ ]:
preview_samples = {}

for record_set_id in record_set_ids:
    print(f"\nPreviewing records for record set @id: {record_set_id}")
    try:
        sample_records = []
        for i, x in enumerate(dataset.records(record_set=record_set_id)):
            sample_records.append(x)
            print(x)
            if i >= 2:
                break
        preview_samples[record_set_id] = sample_records
        if not sample_records:
            print("No records returned.")
    except Exception as e:
        print(f"Could not preview records for {record_set_id}: {e}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. We use record set and field `@id`s discovered in the overview section.

The code below dynamically loops over all discovered record set `@id` values, loads their records, and stores each table in a dictionary keyed by record set `@id`.

In [ ]:
# Extract data from each discovered record set
record_sets = record_set_ids.copy()
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows x {len(df.columns)} columns from record set @id: {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

if dataframes:
    first_record_set_id = next(iter(dataframes))
    print(f"\nExample record set @id: {first_record_set_id}")
    print("Columns:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets could be loaded into DataFrames.")

Because this dataset package may emphasize distributions and metadata over declared top-level record sets, the next cell also inspects the file distributions available in the Croissant package. This can help identify where tabular data may live, even if `recordSet` entries are not exposed at the top level.

In [ ]:
distributions = metadata_json.get("distribution", [])
print(f"Number of distribution entries: {len(distributions)}")

distribution_df = pd.DataFrame(distributions)
display(distribution_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Since Croissant datasets may expose different schemas, this section automatically:
1. picks a loaded record set,
2. identifies numeric columns,
3. filters records based on a quantile threshold,
4. normalizes the selected numeric field,
5. optionally groups by a categorical field.

Whenever possible, use the discovered field names as field identifiers corresponding to the record-set schema.

In [ ]:
if not dataframes:
    print("EDA skipped because no record-set DataFrames were loaded.")
else:
    # Select the first non-empty DataFrame
    selected_record_set_id = None
    selected_df = None
    for rs_id, df in dataframes.items():
        if not df.empty:
            selected_record_set_id = rs_id
            selected_df = df.copy()
            break
    
    if selected_df is None:
        print("EDA skipped because all loaded DataFrames are empty.")
    else:
        print(f"Using record set @id for EDA: {selected_record_set_id}")
        
        numeric_columns = selected_df.select_dtypes(include=np.number).columns.tolist()
        print("Numeric field candidates:")
        print(numeric_columns)
        
        if numeric_columns:
            numeric_field = numeric_columns[0]
            threshold = selected_df[numeric_field].quantile(0.75)
            filtered_df = selected_df[selected_df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold}:")
            display(filtered_df.head())
            
            std_value = filtered_df[numeric_field].std()
            if pd.notna(std_value) and std_value != 0:
                filtered_df[f"{numeric_field}_normalized"] = (
                    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
                )
            else:
                filtered_df[f"{numeric_field}_normalized"] = 0.0
            
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
            
            categorical_columns = selected_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
            group_field = categorical_columns[0] if categorical_columns else None
            
            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped mean of {numeric_field} by {group_field}:")
                display(grouped_df.head())
            else:
                print("No categorical grouping field was available in the selected record set.")
        else:
            print("No numeric fields were found for EDA in the selected record set.")

### Optional schema-aware summary
This helper summarizes each loaded record set by `@id`, row count, column count, and numeric fields. It is useful when a Croissant package contains multiple tables.

In [ ]:
summary_rows = []

for rs_id, df in dataframes.items():
    summary_rows.append({
        "record_set_@id": rs_id,
        "rows": len(df),
        "columns": len(df.columns),
        "numeric_fields": ", ".join(df.select_dtypes(include=np.number).columns.tolist())
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df if not summary_df.empty else pd.DataFrame({"message": ["No loaded record sets to summarize."]}))

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The plotting code below adapts to the first loaded non-empty DataFrame. If at least one numeric field exists, it draws a histogram. If at least two numeric fields exist, it also draws a scatter plot.

In [ ]:
if not dataframes:
    print("Visualization skipped because no record-set DataFrames were loaded.")
else:
    viz_record_set_id = None
    viz_df = None
    for rs_id, df in dataframes.items():
        if not df.empty:
            viz_record_set_id = rs_id
            viz_df = df.copy()
            break
    
    if viz_df is None:
        print("Visualization skipped because all DataFrames are empty.")
    else:
        numeric_columns = viz_df.select_dtypes(include=np.number).columns.tolist()
        print(f"Using record set @id for visualization: {viz_record_set_id}")
        print(f"Numeric columns: {numeric_columns}")
        
        if len(numeric_columns) >= 1:
            plt.figure(figsize=(8, 5))
            sns.histplot(viz_df[numeric_columns[0]].dropna(), kde=True, bins=20)
            plt.title(f"Distribution of {numeric_columns[0]}\nRecord set @id: {viz_record_set_id}")
            plt.xlabel(numeric_columns[0])
            plt.ylabel("Count")
            plt.tight_layout()
            plt.show()
        else:
            print("No numeric columns available for histogram visualization.")
        
        if len(numeric_columns) >= 2:
            plt.figure(figsize=(7, 5))
            sns.scatterplot(data=viz_df, x=numeric_columns[0], y=numeric_columns[1])
            plt.title(f"{numeric_columns[0]} vs {numeric_columns[1]}\nRecord set @id: {viz_record_set_id}")
            plt.tight_layout()
            plt.show()
        else:
            print("At least two numeric columns are required for a scatter plot.")

## 6. Conclusion
This notebook demonstrated how to:
- load a Croissant dataset from its schema URL with `mlcroissant`,
- inspect metadata without treating `dataset.metadata` as a subscriptable dictionary,
- discover record set and field `@id` values dynamically,
- extract records into pandas DataFrames,
- perform basic filtering, normalization, and grouping,
- and create simple visualizations.

For this FAIR^2 package, schema inspection is especially important because the package may expose rich metadata and file distributions even when top-level `recordSet` declarations are sparse or absent. A practical next step is to inspect the referenced distributions in more detail and, if needed, map them to the packaged Parquet resources for deeper table-level analysis.